# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets and their IDs, and list all fields (columns) for each record set, referencing them by their `@id` fields.

In [ ]:
from collections import defaultdict

record_sets = list(dataset.record_sets.values())
print(f"Number of record sets: {len(record_sets)}")
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"Record set name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields (by @id):")
        for field in rs.fields.values():
            print(f"    - {field.id} ({field.name})")
        print("")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for analysis. Record sets and fields are referenced by their `@id`s.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets.values()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")
    print(f"  DataFrame columns (by @id): {list(df.columns)}")
    print("")

# For analysis, select the first record set (or update as appropriate)
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]
    print(f"Selected record set for further analysis: {selected_record_set_id}")
    print("Sample records:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping for numeric fields using columns referenced by their `@id`s.

In [ ]:
# For demonstration, automatically search for numeric fields
import numpy as np

df = dataframes[selected_record_set_id]
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric field candidates detected: {numeric_field_candidates}")

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using {numeric_field_id} as the numeric field for analysis.")

    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a plausible group field (choose the first non-numeric field)
    group_field_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize the data distributions or relationships between fields. Numeric fields and group fields are referenced by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_candidates:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot of numeric field by group if available
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in df.columns:
            plt.figure(figsize=(10, 5))
            df.boxplot(column=numeric_field_id, by=group_field)
            plt.title(f"Boxplot of {numeric_field_id} by {group_field}")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field_id)
            plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library. We:
- Loaded metadata and explored available record sets and their fields by `@id`.
- Loaded data into pandas DataFrames by referencing record sets with their `@id`s.
- Identified and processed numeric fields, filtered data, normalized variables, and examined group-wise statistics by referencing fields through their `@id`s.
- Visualized distributions and relationships in the data.

This workflow provides a reproducible, standards-based approach to exploring Croissant datasets in Python.
